# Expected limit plots vs topology, bound state energy, and lifetime

Reads the limits produced by `sidm/scripts/run_combine_limits.py` (see
`make_datacards.ipynb` for how the datacards are built) and plots the expected 95% CL upper
limit on the signal cross section against the three axes of the signal grid:

* **topology** — `4Mu` or `2Mu2E`;
* **bound state energy** — `m_bound`, the mass of the bound state (200, 500, 800, 1000 GeV);
* **lifetime** — the generated `ctau`, converted to the **average lab-frame $L_{xy}$** with the
  lookup table below, because that is the quantity the detector actually responds to.

Because the signal is normalised to a 1 fb reference cross section, Combine's `r` is directly
the limit on $\sigma$ in fb.

**Plot style** (as requested):

| element | meaning |
|---|---|
| black line | median expected limit, `expected_50` |
| green band | 1$\sigma$ band, `expected_16` to `expected_84` |
| yellow band | 2$\sigma$ band, `expected_2p5` to `expected_97p5` |

log y-axis throughout, log x-axis for $L_{xy}$. This is the standard Brazil-plot convention.

All figures are written to `plots/`.

In [ ]:
import os
import sys
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(1, os.path.join(os.getcwd(), '../../..'))
from sidm.tools import utilities

utilities.set_plot_style()
%matplotlib inline

STUDY_DIR = Path(os.getcwd())
LIMIT_CSV = STUDY_DIR / "limits" / "limits.csv"
PLOT_DIR = STUDY_DIR / "plots"

# One output directory per family of plots.
OUT_DIRS = {
    "brazil_vs_lxy": PLOT_DIR / "brazil_vs_lxy",
    "vs_lxy_by_mass": PLOT_DIR / "vs_lxy_by_mass",
    "vs_lxy_by_topology": PLOT_DIR / "vs_lxy_by_topology",
    "vs_mass": PLOT_DIR / "vs_mass",
    "summary": PLOT_DIR / "summary",
}
for path in OUT_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

print("reading", LIMIT_CSV)
print("writing plots under", PLOT_DIR)

## 1. `ctau` $\to$ average lab-frame $L_{xy}$

Each row of the table is `(m_bound, mzd, ctau_1 ... ctau_5)` in mm, and the five `ctau` columns
correspond in order to the five average lab-frame $L_{xy}$ values in cm. So the mapping is
keyed on the full `(m_bound, mzd, ctau_mm)` triple — the same `ctau` means a different $L_{xy}$
at a different mass, which is exactly why the conversion is needed before masses can be
compared on one axis.

In [ ]:
table_rows = [
    (100, 0.25, 0.02, 0.2, 2, 10, 20),
    (100, 1.2, 0.096, 0.96, 9.6, 48, 96),
    (100, 5, 0.4, 4, 40, 200, 400),
    (150, 0.25, 0.013, 0.13, 1.3, 6.7, 13),
    (150, 1.2, 0.064, 0.64, 6.4, 32, 64),
    (150, 5, 0.27, 2.7, 27, 130, 270),
    (200, 0.25, 0.01, 0.1, 1, 5, 10),
    (200, 1.2, 0.048, 0.48, 4.8, 24, 48),
    (200, 5, 0.2, 2, 20, 100, 200),
    (500, 0.25, 0.004, 0.04, 0.4, 2, 4),
    (500, 1.2, 0.019, 0.19, 1.9, 9.6, 19),
    (500, 5, 0.08, 0.8, 8, 40, 80),
    (800, 0.25, 0.0025, 0.025, 0.25, 1.2, 2.5),
    (800, 1.2, 0.012, 0.12, 1.2, 6, 12),
    (800, 5, 0.05, 0.5, 5, 25, 50),
    (1000, 0.25, 0.002, 0.02, 0.2, 1, 2),
    (1000, 1.2, 0.0096, 0.096, 0.96, 4.8, 9.6),
    (1000, 5, 0.04, 0.4, 4, 20, 40),
]
avg_lab_lxy_cm_values = [0.3, 3.0, 30.0, 150.0, 300.0]

# (m_bound, mzd, ctau_mm) -> average lab-frame Lxy in cm.  The ctau keys are
# rounded to guard against float noise in the sample names (e.g. 0.0096).
LXY_LOOKUP = {
    (float(row[0]), float(row[1]), round(float(ctau), 6)): lxy
    for row in table_rows
    for ctau, lxy in zip(row[2:], avg_lab_lxy_cm_values)
}

def avg_lab_lxy_cm(m_bound, mzd, ctau_mm):
    '''Average lab-frame Lxy in cm for one grid point, or NaN if not tabulated.'''
    return LXY_LOOKUP.get((float(m_bound), float(mzd), round(float(ctau_mm), 6)), np.nan)

print(f"{len(LXY_LOOKUP)} grid points tabulated "
      f"({len(table_rows)} (m_bound, mzd) rows x {len(avg_lab_lxy_cm_values)} lifetimes)")

In [ ]:
limits = pd.read_csv(LIMIT_CSV)

# The datacard filenames call these m_mediator / m_darkphoton; rename to the
# vocabulary of the lookup table.
limits = limits.rename(columns={
    "final_state": "topology",
    "m_mediator": "m_bound",
    "m_darkphoton": "mzd",
    "exp_m2": "expected_2p5",
    "exp_m1": "expected_16",
    "exp": "expected_50",
    "exp_p1": "expected_84",
    "exp_p2": "expected_97p5",
})
limits["lxy_cm"] = [
    avg_lab_lxy_cm(m, z, c)
    for m, z, c in zip(limits.m_bound, limits.mzd, limits.ctau)
]

missing = limits[limits.lxy_cm.isna()]
if len(missing):
    raise ValueError(
        f"{len(missing)} grid points are not in the lookup table:\n"
        + missing[["topology", "m_bound", "mzd", "ctau"]].to_string(index=False)
    )

limits = limits.sort_values(["topology", "m_bound", "mzd", "lxy_cm"]).reset_index(drop=True)
print(f"{len(limits)} limits, all resolved to an Lxy")
print("topologies :", sorted(limits.topology.unique()))
print("m_bound    :", sorted(limits.m_bound.unique()))
print("mzd        :", sorted(limits.mzd.unique()))
print("Lxy [cm]   :", sorted(limits.lxy_cm.unique()))
limits[["topology", "m_bound", "mzd", "ctau", "lxy_cm",
        "expected_2p5", "expected_16", "expected_50",
        "expected_84", "expected_97p5"]].head(10)

## 2. Plot helpers

`BAND_STYLE` is the single place the colour assignment lives. `GRID_RC` shrinks the CMS
style's 26 pt text for the multi-panel figures, where it would otherwise overrun the panels.

In [ ]:
# Standard Brazil-plot convention: green inner 1 sigma, yellow outer 2 sigma.
BAND_STYLE = {
    "one_sigma": {"color": "#00CC00", "label": r"expected $\pm 1\sigma$"},
    "two_sigma": {"color": "#FFCC00", "label": r"expected $\pm 2\sigma$"},
    "median": {"color": "black", "ls": "--", "lw": 2, "label": "median expected"},
}

# The CMS plot style sets font.size to 26, which is right for a single full-size
# figure but illegible once several panels share one canvas.  Multi-panel
# figures are built inside plt.rc_context(GRID_RC) to scale the text down.
GRID_RC = {
    "font.size": 13,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "xtick.minor.size": 3,
    "ytick.minor.size": 3,
}
XLABEL_LXY = r"average lab-frame $L_{xy}$ [cm]"
XLABEL_MASS = r"bound state energy $m_\mathrm{bound}$ [GeV]"
YLABEL = r"95% CL upper limit on $\sigma$ [fb]"
# Shorter form for the overlay plots, whose axis label would otherwise be
# wider than the figure at this plot style's font size.
YLABEL_MEDIAN = r"median limit on $\sigma$ [fb]"

# Distinguishable, colour-blind-safe series colours for the overlay plots.
SERIES_COLORS = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9"]


def draw_brazil(ax, group, x="lxy_cm"):
    '''Median expected limit with its 1 and 2 sigma bands, on a log y-axis.

    The 2 sigma band is drawn first so the narrower 1 sigma band sits on top of
    it rather than being hidden underneath.
    '''
    group = group.sort_values(x)
    ax.fill_between(group[x], group.expected_2p5, group.expected_97p5,
                    **BAND_STYLE["two_sigma"])
    ax.fill_between(group[x], group.expected_16, group.expected_84,
                    **BAND_STYLE["one_sigma"])
    ax.plot(group[x], group.expected_50, **BAND_STYLE["median"])
    ax.set_yscale("log")
    ax.set_ylabel(YLABEL)
    ax.grid(alpha=0.3, which="both")
    return ax


def style_lxy_axis(ax):
    ax.set_xscale("log")
    ax.set_xlabel(XLABEL_LXY)


def save(fig, outdir, name):
    '''Write a figure as both png and pdf, and return the png path.'''
    for ext in ("png", "pdf"):
        fig.savefig(OUT_DIRS[outdir] / f"{name}.{ext}", dpi=150, bbox_inches="tight")
    return OUT_DIRS[outdir] / f"{name}.png"


def label_point(topology, m_bound, mzd):
    return (f"{topology}, $m_\\mathrm{{bound}}$ = {m_bound:g} GeV, "
            f"$m_{{Z_D}}$ = {mzd:g} GeV")

## 3. Expected limit vs lifetime

One figure per (topology, bound state energy, $m_{Z_D}$): the full band structure against
average lab-frame $L_{xy}$.

In [ ]:
made = []
for (topology, m_bound, mzd), group in limits.groupby(["topology", "m_bound", "mzd"]):
    fig, ax = plt.subplots(figsize=(7, 5))
    draw_brazil(ax, group)
    style_lxy_axis(ax)
    ax.set_title(label_point(topology, m_bound, mzd), fontsize=11)
    ax.legend(loc="best", fontsize=9)
    name = f"limit_vs_lxy_{topology}_mbound{m_bound:g}_mzd{mzd:g}".replace(".", "p")
    made.append(save(fig, "brazil_vs_lxy", name))
    plt.close(fig)

print(f"wrote {len(made)} figures to {OUT_DIRS['brazil_vs_lxy']}")

In [ ]:
# Same thing as one overview grid per topology, for reading at a glance.
# Axes are shared and only the outer ones are labelled, so the tick labels have
# room to breathe.
for topology in sorted(limits.topology.unique()):
    subset = limits[limits.topology == topology]
    masses = sorted(subset.m_bound.unique())
    mzds = sorted(subset.mzd.unique())
    with plt.rc_context(GRID_RC):
        fig, axes = plt.subplots(len(mzds), len(masses),
                                 figsize=(4.4 * len(masses), 3.6 * len(mzds)),
                                 squeeze=False, sharex=True, sharey=True,
                                 layout="constrained")
        for (i, mzd), (j, m_bound) in itertools.product(enumerate(mzds), enumerate(masses)):
            ax = axes[i][j]
            group = subset[(subset.m_bound == m_bound) & (subset.mzd == mzd)]
            draw_brazil(ax, group)
            style_lxy_axis(ax)
            ax.set_title(f"$m_\\mathrm{{bound}}$={m_bound:g} GeV, $m_{{Z_D}}$={mzd:g} GeV")
            # only the outer panels carry axis labels
            if j:
                ax.set_ylabel("")
            if i != len(mzds) - 1:
                ax.set_xlabel("")
        axes[0][0].legend(loc="upper right")
        fig.suptitle(f"{topology}: expected limit vs average lab-frame $L_{{xy}}$")
        print(save(fig, "summary", f"grid_brazil_vs_lxy_{topology}"))
        plt.show()

## 4. Dependence on bound state energy

Median expected limits for every bound state energy overlaid on one $L_{xy}$ axis, one figure
per (topology, $m_{Z_D}$). The 1$\sigma$ band of the best-performing mass is kept as a shaded
reference so the spread between masses can be judged against the uncertainty on any one of them.

In [ ]:
for (topology, mzd), subset in limits.groupby(["topology", "mzd"]):
    fig, ax = plt.subplots(figsize=(7.5, 5.5))

    masses = sorted(subset.m_bound.unique())
    best = subset.loc[subset.expected_50.idxmin(), "m_bound"]
    reference = subset[subset.m_bound == best].sort_values("lxy_cm")
    ax.fill_between(reference.lxy_cm, reference.expected_16, reference.expected_84,
                    color=BAND_STYLE["one_sigma"]["color"], alpha=0.45,
                    label=rf"$\pm 1\sigma$, $m_\mathrm{{bound}}$ = {best:g} GeV")

    for colour, m_bound in zip(itertools.cycle(SERIES_COLORS), masses):
        group = subset[subset.m_bound == m_bound].sort_values("lxy_cm")
        ax.plot(group.lxy_cm, group.expected_50, marker="o", ms=5, color=colour,
                label=rf"$m_\mathrm{{bound}}$ = {m_bound:g} GeV")

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(XLABEL_LXY)
    ax.set_ylabel(YLABEL_MEDIAN)
    ax.set_title(f"{topology}, $m_{{Z_D}}$ = {mzd:g} GeV", fontsize=11)
    ax.grid(alpha=0.3, which="both")
    ax.legend(fontsize=9)
    print(save(fig, "vs_lxy_by_mass",
               f"median_vs_lxy_{topology}_mzd{mzd:g}".replace(".", "p")))
    plt.show()

In [ ]:
# The same dependence read the other way round: limit vs bound state energy,
# one line per lifetime.  x is linear here -- only Lxy gets a log axis.
for (topology, mzd), subset in limits.groupby(["topology", "mzd"]):
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    for colour, lxy in zip(itertools.cycle(SERIES_COLORS), sorted(subset.lxy_cm.unique())):
        group = subset[subset.lxy_cm == lxy].sort_values("m_bound")
        ax.plot(group.m_bound, group.expected_50, marker="o", ms=5, color=colour,
                label=rf"$L_{{xy}}$ = {lxy:g} cm")
    ax.set_yscale("log")
    ax.set_xlabel(XLABEL_MASS)
    ax.set_ylabel(YLABEL_MEDIAN)
    ax.set_title(f"{topology}, $m_{{Z_D}}$ = {mzd:g} GeV", fontsize=11)
    ax.grid(alpha=0.3, which="both")
    ax.legend(fontsize=9)
    print(save(fig, "vs_mass", f"median_vs_mbound_{topology}_mzd{mzd:g}".replace(".", "p")))
    plt.show()

## 5. Dependence on topology

`4Mu` and `2Mu2E` compared directly at the same grid point. They are independent measurements
in different signal regions, so this compares the two channels' reach; it is not a combination.

In [ ]:
pairs = list(limits.groupby(["m_bound", "mzd"]))
for (m_bound, mzd), subset in pairs:
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    for colour, topology in zip(SERIES_COLORS, sorted(subset.topology.unique())):
        group = subset[subset.topology == topology].sort_values("lxy_cm")
        ax.fill_between(group.lxy_cm, group.expected_16, group.expected_84,
                        color=colour, alpha=0.2)
        ax.plot(group.lxy_cm, group.expected_50, marker="o", ms=5, color=colour,
                label=f"{topology} (median, $\\pm 1\\sigma$)")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(XLABEL_LXY)
    ax.set_ylabel(YLABEL)
    ax.set_title(rf"$m_\mathrm{{bound}}$ = {m_bound:g} GeV, $m_{{Z_D}}$ = {mzd:g} GeV",
                 fontsize=11)
    ax.grid(alpha=0.3, which="both")
    ax.legend(fontsize=9)
    print(save(fig, "vs_lxy_by_topology",
               f"topology_comparison_mbound{m_bound:g}_mzd{mzd:g}".replace(".", "p")))
    plt.close(fig)

print(f"\nwrote {len(pairs)} figures to {OUT_DIRS['vs_lxy_by_topology']}")

In [ ]:
# Every median limit on one pair of axes, split by topology.  The legend is 12
# entries long, so it goes underneath the panels rather than on top of the data.
with plt.rc_context(GRID_RC):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.6), sharey=True, layout="constrained")
    # 12 series on one axis: colour carries the mass, line style carries mzd, so
    # every combination stays distinguishable (a plain colour cycle repeats).
    mass_colour = dict(zip(sorted(limits.m_bound.unique()), SERIES_COLORS))
    mzd_style = {0.25: "-", 1.2: "--", 5.0: ":"}
    for ax, topology in zip(axes, sorted(limits.topology.unique())):
        subset = limits[limits.topology == topology]
        for (m_bound, mzd), group in subset.groupby(["m_bound", "mzd"]):
            group = group.sort_values("lxy_cm")
            ax.plot(group.lxy_cm, group.expected_50, marker="o", ms=4,
                    color=mass_colour[m_bound], ls=mzd_style[mzd],
                    label=rf"$m_\mathrm{{bound}}$={m_bound:g}, $m_{{Z_D}}$={mzd:g} GeV")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel(XLABEL_LXY)
        ax.set_title(topology)
        ax.grid(alpha=0.3, which="both")
    axes[0].set_ylabel(YLABEL_MEDIAN)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="outside lower center", ncol=4, fontsize=9)
    print(save(fig, "summary", "median_vs_lxy_all_points"))
    plt.show()

In [ ]:
# Median limit over the (bound state energy, Lxy) plane, one panel per (topology, mzd).
# Both axes are categorical here, so the cells are evenly spaced rather than to scale.
topologies = sorted(limits.topology.unique())
mzds = sorted(limits.mzd.unique())

with plt.rc_context(GRID_RC):
    fig, axes = plt.subplots(len(topologies), len(mzds),
                             figsize=(4.8 * len(mzds), 4.0 * len(topologies)),
                             squeeze=False, layout="constrained")
    norm = plt.matplotlib.colors.LogNorm(vmin=limits.expected_50.min(),
                                         vmax=limits.expected_50.max())

    for (i, topology), (j, mzd) in itertools.product(enumerate(topologies), enumerate(mzds)):
        ax = axes[i][j]
        subset = limits[(limits.topology == topology) & (limits.mzd == mzd)]
        table = subset.pivot_table(index="m_bound", columns="lxy_cm", values="expected_50")
        mesh = ax.pcolormesh(np.arange(table.shape[1] + 1), np.arange(table.shape[0] + 1),
                             table.values, norm=norm, cmap="viridis_r", shading="flat")
        ax.set_xticks(np.arange(table.shape[1]) + 0.5, [f"{c:g}" for c in table.columns])
        ax.set_yticks(np.arange(table.shape[0]) + 0.5, [f"{r:g}" for r in table.index])
        ax.tick_params(length=0)
        for y, x in itertools.product(range(table.shape[0]), range(table.shape[1])):
            ax.text(x + 0.5, y + 0.5, f"{table.values[y, x]:.3g}",
                    ha="center", va="center", fontsize=9, color="white")
        ax.set_title(f"{topology}, $m_{{Z_D}}$ = {mzd:g} GeV")
        # only the outer panels carry axis labels
        if i == len(topologies) - 1:
            ax.set_xlabel(XLABEL_LXY)
        if j == 0:
            ax.set_ylabel(XLABEL_MASS)

    fig.colorbar(mesh, ax=axes, label=YLABEL_MEDIAN, shrink=0.75)
    print(save(fig, "summary", "median_limit_map"))
    plt.show()

In [ ]:
# Where the analysis is most and least sensitive
columns = ["topology", "m_bound", "mzd", "ctau", "lxy_cm", "expected_50"]
print("most sensitive points:")
print(limits.nsmallest(10, "expected_50")[columns].to_string(index=False))
print("\nleast sensitive points:")
print(limits.nlargest(5, "expected_50")[columns].to_string(index=False))

summary_csv = PLOT_DIR / "limits_with_lxy.csv"
limits.to_csv(summary_csv, index=False)
print(f"\nwrote the Lxy-annotated limit table to {summary_csv}")